# Lecture 6: Electronic Structure

## Overview
**Questions**
- How do I compute an electronic band structure with DFT?
- How do I compute a density of states?
- How do I interpret these for quantum optics applications?

**Objectives**
- Understand the workflow for band structure calculations
- Compute and plot a band structure along high-symmetry paths
- Understand direct vs indirect bandgap and why it matters for light emission


## Why Electronic Structure Matters for Quantum Optics

The electronic structure of a material determines:
- **Bandgap** — whether the material can emit visible/NIR light
- **Direct/indirect gap** — whether light emission is allowed by momentum conservation
- **Defect levels** — the energy of in-gap states associated with quantum emitters
- **Effective masses** — carrier transport relevant for device engineering

### Key materials and their bandgaps

| Material | Bandgap (eV) | Type | Application |
|----------|-------------|------|-------------|
| Diamond | 5.47 | Indirect | NV centre host |
| hBN | ~6.0 | Indirect (bulk) / ~6.1 (mono.) | V_B emitter |
| GaN | 3.4 | Direct | LEDs, quantum emitter host |
| AlN | 6.0 | Direct | Deep-UV emitters |
| Si | 1.1 | Indirect | Photonic circuits |
| GaAs | 1.4 | Direct | Quantum dots |


## Band Structure Workflow

Computing a band structure requires two DFT calculations:

1. **Self-consistent field (SCF)** — solve for the electron density on a uniform k-mesh
2. **Non-self-consistent (NSCF)** — compute eigenvalues at k-points along high-symmetry paths (using the fixed density from step 1)

ASE provides `BandPath` objects that enumerate high-symmetry paths for any Bravais lattice:


In [ ]:
from ase.build import bulk
from ase.dft.kpoints import get_special_points, bandpath
import numpy as np
import matplotlib.pyplot as plt

# Get the high-symmetry points for the FCC lattice (used by diamond and GaAs)
diamond_C = bulk('C', 'diamond', a=3.57)
path = diamond_C.cell.bandpath('GXWKGLUWLK', npoints=100)

print("High-symmetry path for diamond/zinc-blende FCC:")
print(f"  Path: {path.path}")
print(f"  Number of k-points: {len(path.kpts)}")
print(f"\nSpecial points:")
sp = path.special_points
for label, kpt in sp.items():
    print(f"  {label}: {kpt}")


In [ ]:
# Hexagonal (wurtzite GaN) Brillouin zone path
gan_wz = bulk('GaN', 'wurtzite', a=3.19, c=5.19)
path_hex = gan_wz.cell.bandpath('GMKGALHA', npoints=100)
print("High-symmetry path for wurtzite (hexagonal):")
print(f"  Path: {path_hex.path}")
print(f"  Special points: {list(path_hex.special_points.keys())}")


## Using GPAW for a Band Structure

Below is the template for computing a GaN band structure with GPAW. This requires GPAW to be installed. We show the code pattern, then display a pre-computed result.

```python
from gpaw import GPAW, PW, FermiDirac
from ase.build import bulk

gan = bulk('GaN', 'wurtzite', a=3.19, c=5.19)

# Step 1: SCF on uniform k-mesh
calc = GPAW(mode=PW(600),
            xc='PBE',
            kpts={'size': (8, 8, 6), 'gamma': True},
            occupations=FermiDirac(0.01),
            txt='gan_scf.txt')
gan.calc = calc
gan.get_potential_energy()
calc.write('gan_scf.gpw')

# Step 2: Band structure along Γ→M→K→Γ→A
path = gan.cell.bandpath('GMKGALHA', npoints=100)
calc_bs = GPAW('gan_scf.gpw').fixed_density(
    kpts=path,
    symmetry='off',
    txt='gan_bs.txt')
gan.calc = calc_bs
gan.get_potential_energy()

bs = calc_bs.band_structure()
bs.plot(emin=-6, emax=8, filename='gan_bandstructure.png', show=True)
```


In [ ]:
# Since GPAW is not available here, we demonstrate with a schematic
# band structure for pedagogical purposes.
# In your own work, replace this with your GPAW/QE output.

def gan_schematic_bands():
    # Schematic GaN-like direct bandgap band structure
    kpts = np.linspace(0, 1, 100)

    # Valence band maximum at Γ (k=0.5 in our path)
    Eg = 3.4  # GaN bandgap (eV)

    bands = []
    for i, offset in enumerate([-2.0, -1.5, -0.5, 0.0]):   # 4 valence bands
        vb = offset - 1.5 * np.sin(np.pi * kpts)**2
        bands.append(vb)

    for i, offset in enumerate([0.0, 0.5, 1.5]):             # 3 conduction bands
        cb = Eg + offset + 2.0 * np.sin(np.pi * kpts)**2
        bands.append(cb)

    return kpts, bands

kpts, bands = gan_schematic_bands()

fig, ax = plt.subplots(figsize=(6, 7))
for b in bands:
    ax.plot(kpts, b, 'b-', linewidth=1.5)

ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', label='VBM')
ax.axhline(3.4, color='gray', linewidth=0.8, linestyle=':', label='CBM')
ax.annotate('', xy=(0.5, 3.4), xytext=(0.5, 0),
            arrowprops=dict(arrowstyle='<->', color='red', lw=2))
ax.text(0.52, 1.7, f'Eg = 3.4 eV
(direct)', color='red', fontsize=11)

ax.set_xticks([0, 0.33, 0.5, 0.66, 1.0])
ax.set_xticklabels(['Γ', 'M', 'K', 'Γ', 'A'], fontsize=12)
ax.set_ylabel('Energy (eV)', fontsize=12)
ax.set_title('Schematic GaN band structure
(direct bandgap at Γ)', fontsize=12)
ax.set_ylim(-4, 8)
ax.axvline(0.33, color='lightgray', lw=0.8)
ax.axvline(0.5, color='lightgray', lw=0.8)
ax.axvline(0.66, color='lightgray', lw=0.8)
plt.tight_layout()
plt.show()
print("In a DFT+GPAW calculation, replace this with calc_bs.band_structure().plot()")


## Density of States

The **density of states (DOS)** $g(E)$ counts the number of electronic states per unit energy interval. The **projected DOS (PDOS)** resolves contributions from each atomic species or orbital — very useful for understanding defect states.

For a quantum emitter like the NV centre, we look for:
- In-gap states inside the host bandgap
- Localised character (predominantly on the N and surrounding C atoms)


In [ ]:
# Schematic DOS to illustrate the concept
E = np.linspace(-5, 10, 1000)

def gaussian(E, E0, sigma, weight=1.0):
    return weight * np.exp(-(E - E0)**2 / (2*sigma**2)) / (sigma * np.sqrt(2*np.pi))

# Host DOS (valence band + conduction band)
dos_host = (np.exp(-(E+2)**2/2) * (E < 0) +
            gaussian(E, -1.5, 0.5) * (E < 0) +
            gaussian(E, 5.0, 1.0) * (E > 3.4))

# Defect in-gap state
dos_defect = gaussian(E, 1.5, 0.08, weight=0.3)

fig, ax = plt.subplots(figsize=(8, 5))
ax.fill_between(E, dos_host, alpha=0.4, color='steelblue', label='Host (C)')
ax.fill_between(E, dos_defect, alpha=0.8, color='red', label='Defect state (N+V)')
ax.axvline(0, color='gray', lw=1, ls='--')
ax.axvline(3.4, color='gray', lw=1, ls=':')
ax.annotate('Valence band
maximum', xy=(0, 0.5), xytext=(-3, 0.6),
            arrowprops=dict(arrowstyle='->', color='gray'))
ax.annotate('Conduction band
minimum', xy=(3.4, 0.1), xytext=(5, 0.4),
            arrowprops=dict(arrowstyle='->', color='gray'))
ax.set_xlabel('Energy (eV)', fontsize=12)
ax.set_ylabel('DOS (arb. units)', fontsize=12)
ax.set_title('Schematic PDOS for NV centre in diamond', fontsize=12)
ax.legend(fontsize=11)
ax.set_xlim(-5, 10)
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()


## Key Points

- Electronic band structure requires two DFT steps: SCF then NSCF along k-path
- `cell.bandpath()` generates high-symmetry paths for any Bravais lattice
- Direct bandgap materials are required for efficient light emission
- The PDOS reveals in-gap defect states relevant for quantum emitters
- Standard DFT (PBE) **underestimates** bandgaps; use HSE06 or G₀W₀ for accurate gaps

## Exercise 6.1

Using the schematic DOS plot above as a template, modify the code to represent a **boron vacancy in hBN** (bandgap ~6 eV, in-gap state at ~4 eV). Change the labels and colours accordingly.

## Exercise 6.2 (Research)

Look up the calculated bandgaps of GaN, AlN, and hBN computed with PBE and HSE06 in the literature or Materials Project. By how much does PBE underestimate the gap in each case? Why does this matter for simulating quantum optical transitions?
